In [26]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma 
from langchain_google_genai import GoogleGenerativeAIEmbeddings , ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [13]:
load_dotenv()

embedding_model = GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-2-preview')

model = ChatGoogleGenerativeAI(model = 'gemini-3.1-flash-lite-preview')

In [2]:
loader = PyPDFLoader('DEV RESUME.pdf')

In [3]:
user_document = loader.load()

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500 , 
    chunk_overlap = 0
)

In [14]:
chunk = splitter.split_documents(user_document)

In [16]:
chroma_store = Chroma.from_documents(chunk , embedding_model)

In [18]:
retriver = chroma_store.as_retriever(
    search_type = 'mmr' , 
    search_kwargs = {'k' : 3}
)

In [19]:
user_query = 'What is my skills'

In [20]:
retrive_doc = retriver.invoke(user_query)

In [22]:
text_context = '\n\n'.join(d.page_content for d in retrive_doc)

In [25]:
prompt = PromptTemplate(
    template='''Hey , you are my assistant.\n
    Answer the query from the context , if it is insufficent just tell i don't know , \n
    query --- {query}  \n context -- > {text}    
    '''
)

In [27]:
parser  = StrOutputParser()

In [28]:
chain = prompt | model | parser

In [29]:
result = chain.invoke({'query' : user_query , 'text' : text_context})

In [32]:
print(result)

Based on the context provided, your technical skills are:

*   **Programming Languages:** Python, SQL
*   **Machine Learning & Deep Learning:** Machine Learning, Deep Learning, CNNs, NLP
*   **Frameworks:** TensorFlow, Keras
*   **Libraries & Tools:** NumPy, Pandas, Scikit-learn, Matplotlib, OpenCV, NLTK, Git
